# Neuro_ANN: staged training on Colab

Runs the `training/` pipeline on a Colab GPU: **stage 1 (subthreshold)** then **stage 2 (spiking)**, or the smooth curriculum.
Checkpoints, the shared normalization (`scaler.json`) and logs go to Google Drive, so a disconnected runtime can pick up where it stopped: re-run the stage cell (it always passes `--resume`).

Runtime → Change runtime type → **GPU** before starting.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > GPU"

## 1. Get the code

In [ ]:
BRANCH = "claude/project-thread-8tlrfa"   # use "main" once the PR is merged
!rm -rf /content/Neuro_ANN && git clone -q --depth 1 -b $BRANCH https://github.com/ViktorJOlah/Neuro_ANN.git /content/Neuro_ANN
%cd /content/Neuro_ANN
!pip -q install pyyaml
import tensorflow as tf, keras; print("TF", tf.__version__, "Keras", keras.__version__, "GPUs", tf.config.list_physical_devices("GPU"))

Optional, about 1 minute: CPU/GPU smoke tests on synthetic traces.

In [ ]:
!python -m pytest training/tests -q -p no:warnings

## 2. Data and config

Point the pools at your NEURON outputs (`calibrate.hoc` → `vmi*.txt`). Files are read as they are: header lines are dropped automatically and nothing is written back.
If you have a single mixed file, use `{"all": [...]}` and traces are split into subthreshold/spiking by spike count.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE_CONFIG = "training/configs/l23_two_stage.yaml"   # or l23_curriculum.yaml / ball_two_stage.yaml
RUN_NAME    = "l23_cell01_two_stage"
OUTPUT_DIR  = "/content/drive/MyDrive/V1_DNN/ann_runs"
FILES = {
    "subthreshold": ["/content/drive/MyDrive/V1_DNN/Proof_of_principal/L2_3/vmi_sub.txt"],
    "spiking":      ["/content/drive/MyDrive/V1_DNN/Proof_of_principal/L2_3/vmi3.txt"],
}

import os, yaml
from training.config import load_config, save_config
cfg = load_config(BASE_CONFIG, {"run_name": RUN_NAME, "output_dir": OUTPUT_DIR, "data": {"files": FILES}})
# Opt-in closed-loop robustness (changes model behaviour, off by default):
# cfg["training"].update(state_noise_std=0.01, rollout_steps=8, rollout_loss_weight=0.5)
CFG = f"/content/{RUN_NAME}.yaml"
save_config(cfg, CFG)
print(yaml.safe_dump({k: cfg[k] for k in ("data", "stages")}, sort_keys=False))

## 3. Stage 1: subthreshold

In [ ]:
!python -m training.train --config $CFG --stages subthreshold --resume

## 4. Stage 2: spiking (starts from the best stage-1 weights, same normalization)

In [ ]:
!python -m training.train --config $CFG --stages spiking --resume

For the curriculum config, run its stages the same way (`--stages subthreshold`, then `--stages curriculum`), or simply omit `--stages` to run everything in order.

## 5. Learning curves and closed-loop scores

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
run = os.path.join(OUTPUT_DIR, RUN_NAME)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for s in cfg["stages"]:
    h = os.path.join(run, s["name"], "history.csv")
    if not os.path.exists(h): continue
    df = pd.read_csv(h)
    axes[0].semilogy(df.epoch, df.train_loss, label=f'{s["name"]} train')
    axes[0].semilogy(df.epoch, df.monitor, "--", label=f'{s["name"]} val (monitor)')
    for c in [c for c in df.columns if c.startswith("cl_") and c.endswith(("var_explained_sub", "spike_f1"))]:
        d = df[["epoch", c]].dropna(); axes[1].plot(d.epoch, d[c], "o-", label=f'{s["name"]} {c[3:]}')
    ev = os.path.join(run, s["name"], "eval_test.json")
    if os.path.exists(ev): print(s["name"], json.dumps(json.load(open(ev)), indent=1))
axes[0].set_title("one-step loss"); axes[1].set_title("closed-loop (val traces)"); axes[1].set_ylim(-0.2, 1.05)
for a in axes: a.legend(fontsize=7); a.set_xlabel("epoch")
plt.show()

## 6. Closed-loop simulation of held-out test traces

In [ ]:
from training.trainer import Trainer
import numpy as np
from training.evaluate import rollout
tr = Trainer(cfg, resume=True)
last = [s["name"] for s in cfg["stages"] if os.path.exists(os.path.join(run, s["name"], "best.weights.h5"))][-1]
tr.model.load_weights(os.path.join(run, last, "best.weights.h5"))
pool = "spiking" if "spiking" in tr.ds.pools else tr.ds.pools[0]
ids = tr.ds.trace_ids(pool, "test")[:3]
pred, true = rollout(tr.model, tr.ds, ids, steps=4000)
vi = tr.ds.state_columns.index(tr.ds.vm_col); lo, hi = tr.ds.scaler.mins[vi], tr.ds.scaler.maxs[vi]
fig, axes = plt.subplots(len(ids), 1, figsize=(13, 2.5 * len(ids)), sharex=True)
for a, p, t in zip(np.atleast_1d(axes), pred, true):
    a.plot(t[:, vi] * (hi - lo) + lo, "k", lw=0.8, label="NEURON"); a.plot(p[:, vi] * (hi - lo) + lo, "r", lw=0.8, label="ANN (closed loop)")
    a.set_ylabel("Vm (mV)")
np.atleast_1d(axes)[0].legend(); plt.xlabel("sample"); plt.show()

## 7. Using the trained model elsewhere

* `<run>/<stage>/best_model.h5` loads with `keras.models.load_model(path, compile=False)` (also in the old scripts).
* `<run>/scaler.json` holds the normalization: `x_norm = (x - min) / (max - min)` per state column, after the preprocessing listed in it. Use the same values in simulation code instead of hard-coded constants.
* The model outputs `(batch, repeats, n_state)`; the prediction is the mean over axis 1, as in the original `predict.py`.